In [1]:
from pathlib import Path
import re
import time
import math
import gc
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# =========================================================
# 경로/설정 (여기만 먼저 수정해서 바로 실행)
# =========================================================
IMG_ROOT = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_filter(1)")   # 1088개 이미지 폴더 루트 (하위폴더 재귀검색)
OUT_ROOT = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_coretest(2)_1088")

# 처리 대상 bed 범위
BED_MIN = 0
BED_MAX = 95

# 이미지 확장자
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

# 병렬 처리
MAX_WORKERS = 8

# 이미지 크기가 너무 크면 속도 저하가 있으므로, 특징 계산용 다운스케일 최대 한계
MAX_SIDE_FOR_FEATURE = 768

# 후보 탐색 관련 파라미터
CANDIDATE_TOPK = 3
LOCAL_R_LIST = [15, 25, 35]       # 로컬 통계 반경
DENSITY_R = 25                    # 조밀도 계산 반경
CIRCLE_RADIUS = 18                # 출력용 원 반경
CIRCLE_THICKNESS = 5              # 굵은 주황색
CIRCLE_COLOR_BGR = (0, 140, 255)  # 주황색 (BGR)

# 점수 가중치 (필요시 조절)
W_DT = 0.34
W_CENTER = 0.22
W_DENSITY = 0.18
W_DARK = 0.10
W_GRAD = 0.08
W_SHARPNESS = 0.08

# 저장 파일명
CSV_NAME = "candidate_metrics_bedall.csv"
LOG_NAME = "run_log.txt"

# =========================================================
# 폴더 생성
# =========================================================
VIS_DIR = OUT_ROOT / "vis_core_candidate"
HEAT_DIR = OUT_ROOT / "heatmap_optional"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)
HEAT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# =========================================================
# 유틸
# =========================================================
def now_str():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def format_seconds(sec):
    sec = max(0, int(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def write_log(msg):
    print(msg)
    with open(OUT_ROOT / LOG_NAME, "a", encoding="utf-8") as f:
        f.write(msg + "\n")


def recursive_find_images(root: Path):
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]


def parse_bed_num(path: Path):
    m = re.search(r"bed(\d{2})", path.name.lower())
    if m:
        return int(m.group(1))
    return None


def filter_bed_range(paths, bed_min=0, bed_max=10):
    out = []
    for p in paths:
        bed_num = parse_bed_num(p)
        if bed_num is not None and bed_min <= bed_num <= bed_max:
            out.append(p)
    return sorted(out)


def resize_with_scale(img, max_side=768, is_mask=False):
    h, w = img.shape[:2]
    scale = 1.0
    if max(h, w) > max_side:
        scale = max_side / max(h, w)
        new_w = max(1, int(round(w * scale)))
        new_h = max(1, int(round(h * scale)))
        interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_AREA
        img = cv2.resize(img, (new_w, new_h), interpolation=interp)
    return img, scale


def keep_largest_component(mask_u8):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num_labels <= 1:
        return mask_u8
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = 1 + np.argmax(areas)
    out = np.zeros_like(mask_u8)
    out[labels == largest_idx] = 255
    return out


def make_mask_from_black_bg(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 완전 검정 배경 가정 + 약간의 여유
    mask = (gray > 8).astype(np.uint8) * 255

    # 작은 구멍 메우기 / 노이즈 제거
    kernel1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel2)
    mask = keep_largest_component(mask)
    return mask


def get_centroid(mask_u8):
    ys, xs = np.where(mask_u8 > 0)
    if len(xs) == 0:
        return None
    return float(xs.mean()), float(ys.mean())


def normalized_center_map(h, w, cx, cy, valid_mask):
    yy, xx = np.mgrid[0:h, 0:w]
    dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    vals = dist[valid_mask > 0]
    if len(vals) == 0:
        out = np.zeros((h, w), np.float32)
        return out
    maxd = vals.max() if vals.max() > 0 else 1.0
    center_score = 1.0 - (dist / maxd)
    center_score = np.clip(center_score, 0, 1).astype(np.float32)
    center_score[valid_mask == 0] = 0
    return center_score


def normalize_on_mask(arr, mask_u8):
    vals = arr[mask_u8 > 0]
    out = np.zeros_like(arr, dtype=np.float32)
    if len(vals) == 0:
        return out
    vmin = float(vals.min())
    vmax = float(vals.max())
    if vmax - vmin < 1e-8:
        out[mask_u8 > 0] = 0
        return out
    out = (arr.astype(np.float32) - vmin) / (vmax - vmin)
    out = np.clip(out, 0, 1)
    out[mask_u8 == 0] = 0
    return out.astype(np.float32)


def local_mean_std(img_gray, ksize):
    f = img_gray.astype(np.float32)
    mean = cv2.blur(f, (ksize, ksize))
    mean2 = cv2.blur(f * f, (ksize, ksize))
    var = np.maximum(mean2 - mean * mean, 0)
    std = np.sqrt(var)
    return mean, std


def local_fill_ratio(mask_u8, radius):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (radius * 2 + 1, radius * 2 + 1)).astype(np.float32)
    kernel /= kernel.sum()
    m = (mask_u8 > 0).astype(np.float32)
    ratio = cv2.filter2D(m, -1, kernel)
    return ratio.astype(np.float32)


def gradient_maps(gray_u8):
    gx = cv2.Sobel(gray_u8, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_u8, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)
    lap = cv2.Laplacian(gray_u8, cv2.CV_32F, ksize=3)
    return mag.astype(np.float32), lap.astype(np.float32)


def extract_topk_candidates(score_map, mask_u8, topk=3, min_distance=20):
    sm = score_map.copy()
    sm[mask_u8 == 0] = -1
    candidates = []

    for _ in range(topk):
        flat_idx = int(np.argmax(sm))
        y, x = np.unravel_index(flat_idx, sm.shape)
        best = float(sm[y, x])
        if best < 0:
            break
        candidates.append((x, y, best))

        yy, xx = np.ogrid[:sm.shape[0], :sm.shape[1]]
        dist2 = (xx - x) ** 2 + (yy - y) ** 2
        sm[dist2 <= min_distance * min_distance] = -1

    return candidates


def safe_crop(arr, x, y, r):
    h, w = arr.shape[:2]
    x1 = max(0, x - r)
    y1 = max(0, y - r)
    x2 = min(w, x + r + 1)
    y2 = min(h, y + r + 1)
    return arr[y1:y2, x1:x2], x1, y1, x2, y2


def compute_candidate_metrics(x, y, score_map, dt_norm, center_norm, density_map, dark_map, grad_norm,
                              sharp_norm, gray_u8, mask_u8, centroid_xy, topk_candidates):
    h, w = gray_u8.shape[:2]
    cx, cy = centroid_xy
    row = {}

    row["candidate_x"] = int(x)
    row["candidate_y"] = int(y)
    row["candidate_score"] = float(score_map[y, x])
    row["dt_norm"] = float(dt_norm[y, x])
    row["center_norm"] = float(center_norm[y, x])
    row["density_norm"] = float(density_map[y, x])
    row["dark_norm"] = float(dark_map[y, x])
    row["grad_norm"] = float(grad_norm[y, x])
    row["sharpness_norm"] = float(sharp_norm[y, x])

    dist_centroid = math.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    diag = math.sqrt(h * h + w * w)
    row["dist_to_centroid_px"] = float(dist_centroid)
    row["dist_to_centroid_normdiag"] = float(dist_centroid / diag)

    row["centroid_x"] = float(cx)
    row["centroid_y"] = float(cy)

    for r in LOCAL_R_LIST:
        patch_gray, x1, y1, x2, y2 = safe_crop(gray_u8, x, y, r)
        patch_mask = mask_u8[y1:y2, x1:x2]
        valid = patch_mask > 0

        if np.any(valid):
            vals = patch_gray[valid]
            row[f"local_mean_r{r}"] = float(vals.mean())
            row[f"local_std_r{r}"] = float(vals.std())
            row[f"local_min_r{r}"] = float(vals.min())
            row[f"local_max_r{r}"] = float(vals.max())
            row[f"local_p10_r{r}"] = float(np.percentile(vals, 10))
            row[f"local_p90_r{r}"] = float(np.percentile(vals, 90))
            row[f"local_range_r{r}"] = float(vals.max() - vals.min())
            row[f"local_fill_ratio_r{r}"] = float(valid.mean())
        else:
            row[f"local_mean_r{r}"] = np.nan
            row[f"local_std_r{r}"] = np.nan
            row[f"local_min_r{r}"] = np.nan
            row[f"local_max_r{r}"] = np.nan
            row[f"local_p10_r{r}"] = np.nan
            row[f"local_p90_r{r}"] = np.nan
            row[f"local_range_r{r}"] = np.nan
            row[f"local_fill_ratio_r{r}"] = np.nan

    # 후보 간 점수 차이
    row["top1_score"] = float(topk_candidates[0][2]) if len(topk_candidates) > 0 else np.nan
    row["top2_score"] = float(topk_candidates[1][2]) if len(topk_candidates) > 1 else np.nan
    row["top3_score"] = float(topk_candidates[2][2]) if len(topk_candidates) > 2 else np.nan
    row["top1_top2_gap"] = float(topk_candidates[0][2] - topk_candidates[1][2]) if len(topk_candidates) > 1 else np.nan
    row["top1_top3_gap"] = float(topk_candidates[0][2] - topk_candidates[2][2]) if len(topk_candidates) > 2 else np.nan

    # confidence 제안값 (0~1)
    gap12 = 0.0 if len(topk_candidates) <= 1 else max(0.0, topk_candidates[0][2] - topk_candidates[1][2])
    confidence = (
        0.45 * row["candidate_score"] +
        0.20 * min(1.0, gap12 * 2.0) +
        0.15 * row["dt_norm"] +
        0.10 * row["density_norm"] +
        0.10 * row["center_norm"]
    )
    confidence = float(max(0.0, min(1.0, confidence)))
    row["confidence_auto"] = confidence

    return row


def save_visualization(img_bgr, x, y, out_path: Path, text_lines=None):
    vis = img_bgr.copy()
    cv2.circle(vis, (int(x), int(y)), CIRCLE_RADIUS, CIRCLE_COLOR_BGR, CIRCLE_THICKNESS, lineType=cv2.LINE_AA)
    cv2.circle(vis, (int(x), int(y)), 3, CIRCLE_COLOR_BGR, -1, lineType=cv2.LINE_AA)

    if text_lines:
        y0 = 28
        for line in text_lines:
            cv2.putText(vis, line, (10, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 3, cv2.LINE_AA)
            cv2.putText(vis, line, (10, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (40, 40, 40), 1, cv2.LINE_AA)
            y0 += 26

    cv2.imwrite(str(out_path), vis)


def process_one_image(img_path: Path):
    try:
        img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            return {"file_path": str(img_path), "status": "read_fail"}

        orig_h, orig_w = img_bgr.shape[:2]

        work_bgr, scale = resize_with_scale(img_bgr, max_side=MAX_SIDE_FOR_FEATURE, is_mask=False)
        gray_u8 = cv2.cvtColor(work_bgr, cv2.COLOR_BGR2GRAY)
        mask_u8 = make_mask_from_black_bg(work_bgr)

        if int(mask_u8.sum()) == 0:
            return {
                "file_path": str(img_path),
                "status": "empty_mask",
                "orig_h": orig_h,
                "orig_w": orig_w,
            }

        centroid = get_centroid(mask_u8)
        if centroid is None:
            return {
                "file_path": str(img_path),
                "status": "no_centroid",
                "orig_h": orig_h,
                "orig_w": orig_w,
            }
        cx, cy = centroid

        # 1) 거리변환
        dt = cv2.distanceTransform(mask_u8, cv2.DIST_L2, 5)
        dt_norm = normalize_on_mask(dt, mask_u8)

        # 2) 중심성
        center_norm = normalized_center_map(mask_u8.shape[0], mask_u8.shape[1], cx, cy, mask_u8)

        # 3) 조밀도
        density_map = local_fill_ratio(mask_u8, DENSITY_R)
        density_norm = normalize_on_mask(density_map, mask_u8)

        # 4) 어두움 / 명암
        local_mean_31, local_std_31 = local_mean_std(gray_u8, 31)
        dark_raw = (255.0 - local_mean_31).astype(np.float32)
        dark_norm = normalize_on_mask(dark_raw, mask_u8)

        # 5) 그래디언트 / sharpness
        grad_mag, lap = gradient_maps(gray_u8)
        grad_norm = normalize_on_mask(grad_mag, mask_u8)
        sharp_raw = np.abs(lap)
        sharp_norm = normalize_on_mask(sharp_raw, mask_u8)

        # 최종 후보 점수
        score_map = (
            W_DT * dt_norm +
            W_CENTER * center_norm +
            W_DENSITY * density_norm +
            W_DARK * dark_norm +
            W_GRAD * grad_norm +
            W_SHARPNESS * sharp_norm
        ).astype(np.float32)
        score_map[mask_u8 == 0] = 0

        # 후보 추출
        topk = extract_topk_candidates(score_map, mask_u8, topk=CANDIDATE_TOPK, min_distance=20)
        if len(topk) == 0:
            return {
                "file_path": str(img_path),
                "status": "no_candidate",
                "orig_h": orig_h,
                "orig_w": orig_w,
            }

        x, y, _ = topk[0]

        row = {
            "file_path": str(img_path),
            "file_name": img_path.name,
            "parent_folder": str(img_path.parent),
            "bed_num": parse_bed_num(img_path),
            "orig_h": orig_h,
            "orig_w": orig_w,
            "work_h": work_bgr.shape[0],
            "work_w": work_bgr.shape[1],
            "resize_scale": float(scale),
            "mask_area_px": int((mask_u8 > 0).sum()),
            "status": "ok",
        }

        row.update(
            compute_candidate_metrics(
                x=x,
                y=y,
                score_map=score_map,
                dt_norm=dt_norm,
                center_norm=center_norm,
                density_map=density_norm,
                dark_map=dark_norm,
                grad_norm=grad_norm,
                sharp_norm=sharp_norm,
                gray_u8=gray_u8,
                mask_u8=mask_u8,
                centroid_xy=(cx, cy),
                topk_candidates=topk,
            )
        )

        # 원본 크기 기준 좌표로 복원해서 시각화
        x_orig = int(round(x / scale)) if scale > 0 else int(x)
        y_orig = int(round(y / scale)) if scale > 0 else int(y)
        row["candidate_x_orig"] = x_orig
        row["candidate_y_orig"] = y_orig

        vis_name = img_path.stem + "__core_candidate.png"
        vis_path = VIS_DIR / vis_name

        text_lines = [
            f"bed={row['bed_num']}  conf={row['confidence_auto']:.3f}",
            f"score={row['candidate_score']:.3f}  dt={row['dt_norm']:.3f}",
            f"center={row['center_norm']:.3f}  density={row['density_norm']:.3f}",
            f"dark={row['dark_norm']:.3f}  grad={row['grad_norm']:.3f}",
        ]
        save_visualization(img_bgr, x_orig, y_orig, vis_path, text_lines=text_lines)
        row["vis_path"] = str(vis_path)

        # 라벨링용 빈 컬럼 미리 추가
        row["core_label"] = np.nan      # 사용자가 0/1/2 입력
        row["label_confidence"] = np.nan
        row["memo"] = ""

        del img_bgr, work_bgr, gray_u8, mask_u8, dt, dt_norm, center_norm
        del density_map, density_norm, local_mean_31, local_std_31, dark_raw, dark_norm
        del grad_mag, lap, grad_norm, sharp_raw, sharp_norm, score_map
        gc.collect()

        return row

    except Exception as e:
        return {
            "file_path": str(img_path),
            "status": f"error: {repr(e)}",
        }



In [3]:

# =========================================================
# 메인
# =========================================================
def main():
    start_time = time.time()
    write_log("=" * 80)
    write_log(f"[START] {now_str()}")
    write_log(f"IMG_ROOT   : {IMG_ROOT}")
    write_log(f"OUT_ROOT   : {OUT_ROOT}")
    write_log(f"BED RANGE  : bed{BED_MIN:02d} ~ bed{BED_MAX:02d}")
    write_log(f"MAX_WORKERS: {MAX_WORKERS}")

    all_imgs = recursive_find_images(IMG_ROOT)
    target_imgs = filter_bed_range(all_imgs, BED_MIN, BED_MAX)

    write_log(f"전체 이미지 수(재귀검색): {len(all_imgs):,}")
    write_log(f"bed00~bed95 대상 이미지 수: {len(target_imgs):,}")

    if len(target_imgs) == 0:
        write_log("대상 이미지가 없습니다. 경로/파일명을 확인하세요.")
        return

    rows = []
    done = 0
    total = len(target_imgs)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(process_one_image, p): p for p in target_imgs}

        pbar = tqdm(total=total, desc="core candidate 추출", unit="img")
        for fut in as_completed(futures):
            row = fut.result()
            rows.append(row)
            done += 1

            elapsed = time.time() - start_time
            avg = elapsed / done if done > 0 else 0
            remain = avg * (total - done)
            pbar.set_postfix_str(f"elapsed={format_seconds(elapsed)}, ETA={format_seconds(remain)}")
            pbar.update(1)

        pbar.close()

    df = pd.DataFrame(rows)

    sort_cols = [c for c in ["bed_num", "file_name"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)

    csv_path = OUT_ROOT / CSV_NAME
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    elapsed_all = time.time() - start_time
    ok_n = int((df["status"] == "ok").sum()) if "status" in df.columns else 0
    err_n = len(df) - ok_n

    write_log(f"CSV 저장 완료: {csv_path}")
    write_log(f"정상 처리: {ok_n:,}")
    write_log(f"오류/제외: {err_n:,}")
    write_log(f"총 소요시간: {format_seconds(elapsed_all)}")
    write_log(f"[END] {now_str()}")
    write_log("=" * 80)

    print("\n완료.")
    print(f"- CSV : {csv_path}")
    print(f"- 시각화 폴더 : {VIS_DIR}")
    print("- 이후 CSV에서 core_label(0/1/2), label_confidence, memo를 직접 입력해서 라벨링하면 됨")



In [4]:

if __name__ == "__main__":
    main()


[START] 2026-03-15 15:46:53
IMG_ROOT   : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_filter(1)
OUT_ROOT   : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_coretest(2)_1088
BED RANGE  : bed00 ~ bed95
MAX_WORKERS: 8
전체 이미지 수(재귀검색): 1,088
bed00~bed10 대상 이미지 수: 1,088


core candidate 추출:   0%|          | 0/1088 [00:00<?, ?img/s]

CSV 저장 완료: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_coretest(2)_1088/candidate_metrics_bedall.csv
정상 처리: 1,088
오류/제외: 0
총 소요시간: 00:02:55
[END] 2026-03-15 15:49:49

완료.
- CSV : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_coretest(2)_1088/candidate_metrics_bedall.csv
- 시각화 폴더 : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_coretest(2)_1088/vis_core_candidate
- 이후 CSV에서 core_label(0/1/2), label_confidence, memo를 직접 입력해서 라벨링하면 됨
